# 02 - Patient Similarity Search

## LLM-Assisted Patient Similarity Using Synthetic Healthcare Data

This notebook implements the patient retrieval component of the project. The goalis to identify patients with similar clinical profiles before providing those records as context to a Large Language Model (LLM).

Patient similarity is estimated using a combination of:
- Clinical conditions represented with TF-IDF
- Patient age
- Gender
- Healthcare encounter count
- Cosine similarity for patient-to-patient comparison

The retrieved patients will later be used to evaluate different LLM prompting and reasoning strategies, including zero-shot prompting, few-shot/in-context learning, Chain-of-Thought, and Tree-of-Thought.

## 1. Project Setup

The reusable implementation for patient similarity is in `src/similarity.py`, while this notebook demonstrates the retrieval workflow and examines its results.

Because the notebook is stored inside the `notebooks/` directory, the project root is added to Python's import path so that modules in `src/` can be imported.

In [1]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd()

if NOTEBOOK_DIR.name == "notebooks":
    PROJECT_ROOT = NOTEBOOK_DIR.parent
else:
    PROJECT_ROOT = NOTEBOOK_DIR

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

### Import Preprocessing Functions

In [2]:
from src.preprocessing import load_patient_master
from src.patient_summary import generate_patient_summary

from src.similarity import (
    build_feature_matrix,
    find_similar_patients,
     explain_similarity_results
)

### Define Data path

In [3]:
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

## 2. Load the Processed Patient Dataset

Milestone 1 transformed the original Synthea tables into a patient-level master dataset. Each row represents one synthetic patient and contains demographic information, clinical conditions, medication history, and healthcare utilization.

The processed pickle file is used here because it preserves the condition and medicaiton columns as Python lists.

In [4]:
patient_df = load_patient_master(
    PROCESSED_DATA_DIR
)

print(patient_df.shape)
patient_df.head()

(12352, 10)


,PATIENT,AGE,GENDER,RACE,ETHNICITY,HEALTHCARE_EXPENSES,HEALTHCARE_COVERAGE,CONDITIONS,MEDICATIONS,ENCOUNTER_COUNT
0,f0f3bc8d-ef38-49ce-a2bd-dfdda982b271,8,M,white,nonhispanic,8446.49,1499.08,"[Otitis media, Fever (finding), Suspected COVI...","[Amoxicillin 250 MG Oral Capsule, Acetaminophe...",5
1,067318a4-db8f-447f-8b6e-f2f61e9baaa5,9,F,white,nonhispanic,89893.40,1845.72,"[Sprain of ankle, Cough (finding), Sputum find...","[Acetaminophen 160 MG Chewable Tablet, Penicil...",5
2,ae9efba3-ddc4-43f9-a781-f72019388548,34,M,white,nonhispanic,577445.86,3528.84,"[Hypertension, Viral sinusitis (disorder), Hea...",[amLODIPine 5 MG / Hydrochlorothiazide 12.5 MG...,13
3,199c586f-af16-4091-9998-ee4cfc02ee7a,22,F,white,nonhispanic,336701.72,2705.64,"[Cough (finding), Sputum finding (finding), Na...",[Jolivette 28 Day Pack],3
4,353016ea-a0ff-4154-85bb-1cf8b6cedf20,29,M,white,nonhispanic,484076.34,3043.04,"[Nasal congestion (finding), Cough (finding), ...",[],1


## 3. Build the Patient Feature Representation

Patient similarity requires converting each patient into a numerical feature vector. Two types of information in this representation:

**Clinical features:** Patient condition descriptions are converted into TF-IDF vectors. TF-IDF assigns greater importance to terms that help distinguish one patient's condition profile from others in the dataset.

**Structured features:** Age and encounter count are standardized so that their different numerical scales do not dominate similarity calculation. Gender is represented using one-hot encoding.

Combining clinical and structured features allos similarity to reflect both a patient's medical conditions and basic characteristics.

In [5]:
patient_df, feature_matrix, artifacts = build_feature_matrix(patient_df)

print("Patients:", len(patient_df))
print("Feature matrix:", feature_matrix.shape)

Patients: 12352
Feature matrix: (12352, 262)


### Feature Matrix

The resulting feature matrix contains **12,352 patients represented by 262 features**. Most features originate from the TF-IDF representation of clinical conditions, while the remaining features represent age, encounter utilization, and gender.

Rather than constructing a full patient-by-patient similarity matrix, similarity will be calculated only when a query patient is selected. This reduces unnecessary computation and memory usage.

## 4. Select a Query Patient

A single patient is selected as the query patient for the initial retrieval experiment. The objective is to identify other patients whoe clinical and structured feature representations are most similar to this patient's profile.

For this initial validation, patient index `0` is used because the patient's COVID-19-related condition history provides a clear example for assessing whether the retrieved neighbors are clinically reasonable.

In [6]:
query_index = 0

query_patient = patient_df.iloc[query_index]

print("Age:", query_patient["AGE"])
print("Gender:", query_patient["GENDER"])
print("Conditions:")
print(query_patient["CONDITIONS"])

Age: 8
Gender: M
Conditions:
['Otitis media', 'Fever (finding)', 'Suspected COVID-19', 'COVID-19']


### Query Patient Profile

The query patient is a pediatric male with a history that includes otitis media, fever, suapected COVID-19, and confirmed COVID-19. This combination provides several clinically meaningful characteristics that can be compared with other synthetic patients.

A useful retrieval system should prioritize patients with overlapping COVID-19-related conditions and similar demographic or healthcare-utilization characteristics.

## 5. Retrieve the Most Similar Patients

Cosine similarity is used to compare the query patient's feature vector with every other patient in the dataset.

A cosine similarity score closer to **1.0** indicates that two patient vectors have highly similar feature patterns. The query patient is excluded from the results, and the three highest-scoring remaining patients are returned.

For this project, `k = 3` is used to keep the retrieved context concise enough for later LLM prompting experiments.

In [7]:
similar_patients = find_similar_patients(
    patient_df,
    feature_matrix,
    query_index=query_index,
    k=3
)

similar_patients[
    [
        "PATIENT",
        "SIMILARITY_SCORE",
        "AGE",
        "GENDER",
        "CONDITIONS",
        "ENCOUNTER_COUNT"
    ]
]

,PATIENT,SIMILARITY_SCORE,AGE,GENDER,CONDITIONS,ENCOUNTER_COUNT
6162,ea68e771-211c-4e79-bfc8-fa2bb394d569,0.999940,9,M,"[Otitis media, Fever (finding), Suspected COVI...",5
3269,30907394-52a9-4625-bf71-4cba86941274,0.994404,7,M,"[Otitis media, Cough (finding), Fever (finding...",5
2940,c1c96726-ea06-414f-82d8-9b24b923cf55,0.994206,9,M,"[Otitis media, Cough (finding), Fever (finding...",4


In [8]:
print(patient_df.loc[0, "CONDITIONS"])

['Otitis media', 'Fever (finding)', 'Suspected COVID-19', 'COVID-19']


## 6. Examine Shared Clinical Conditions

A high numerical similarity score does not automatically guarantee that a retireved patient is clinically meaningful. Therefore, the retrieved neighbors are examined to determine which conditions they share with the query patient.

This step improves the interpretability of the retrieval system and provides a simple sanity check before the retrieved records are supplied to an LLM.

In [9]:
comparison_df = explain_similarity_results(
    patient_df,
    similar_patients,
    query_index,
)

comparison_df[
    [
        "SIMILARITY_SCORE",
        "AGE",
        "GENDER",
        "ENCOUNTER_COUNT",
        "SHARED_CONDITIONS",
    ]
]

,SIMILARITY_SCORE,AGE,GENDER,ENCOUNTER_COUNT,SHARED_CONDITIONS
0,0.999940,9,M,5,"[COVID-19, Fever (finding), Otitis media, Susp..."
1,0.994404,7,M,5,"[COVID-19, Fever (finding), Otitis media, Susp..."
2,0.994206,9,M,4,"[COVID-19, Fever (finding), Otitis media, Susp..."


### Retrieval Results

The three retrieved patients have cosine similarity score above **0.99**, indicating highly similar feature representation.

The results also show that the similarity is clinically interpretable rather than being based only on demographic characteristics. All three retrieved patients share important conditions with the query patient, including:

- COVID-19
- Suspected COVID-19
- Fever
- Otitis media

The patients are also similar in age, gender, and encounter frequency. These findings suggest that the retrieval method successfully identified patients with closely related synthetic clinical profiles.

The extremely high similarity scores are plausible because Synthea generates synthetic patients from common disease progression patterns, which can result in patients with nearly identical combinations of conditions and healthcare utilization.

In [10]:
query_patient = patient_df.iloc[query_index]

print("QUERY PATIENT")
print("-" * 50)
print(f"Age: {query_patient['AGE']}")
print(f"Gender: {query_patient['GENDER']}")
print(f"Encounters: {query_patient['ENCOUNTER_COUNT']}")
print("Conditions")

for condition in query_patient["CONDITIONS"]:
    print(f"    - {condition}")

QUERY PATIENT
--------------------------------------------------
Age: 8
Gender: M
Encounters: 5
Conditions
    - Otitis media
    - Fever (finding)
    - Suspected COVID-19
    - COVID-19


## 7. Convert Structured Patient Data to Text

LLMs operate primarily on textual input rather than numerical feature vectors. Therefore, the structured patient profile containing demographics, conditions, medications, and healthcare utilization.

This transformation creates the bridge between the patient retrieval pipeline and the prompt-engineering experiments in the next milestone. Importantly, the data used in this project are synthetic Synthea records rather than real patient records.



In [11]:
query_summary = generate_patient_summary(patient_df.iloc[query_index])

print(query_summary)


Patient Profile

Age: 8
Gender: M
Race: white
Ethnicity: nonhispanic

Encounter Count: 5

Conditions:
- Otitis media
- Fever (finding)
- Suspected COVID-19
- COVID-19

Medications:
- Amoxicillin 250 MG Oral Capsule
- Acetaminophen 160 MG Chewable Tablet



The generated profile preserves the clinically relevant information from the structured dataset while presenting it in a format that can be incorporated consistently into different prompt templates.

Using the same underlying patient information across prompting strategies will allow the subsequent experiments to compare the effects of prompt design rather than differences in input data.

## Milestone 2 Summary

This milestone established the retrieval component of the LLM-assisted patient similarity pipeline.

The workflow:

1. Loaded the processed Synthea patient dataset.
2. Represented clinical conditions using TF-IDF.
3. Combined clinical features with age, gender, and encounter utilization.
4. Used cosine similarity to retrieve the three most similar patients.
5. Examined shared conditions to validate the clinical relevance of the retrieved neighbors.
6. Converted structured patient information into text suitable for LLM prompting.

The retrieved patients demonstrated strong overlap with the query patient's COVID-19-related clinical profile, providing meaningful context for the next stage of the project.

### Next Step: LLM Prompt Engineering

Milestone 3 will use the query patient and retrieved similar patients to investigate how different prompting strategies affect an LLM's ability to explain clinical similarity.

The experiments will compare:

- Zero-shot prompting
- Few-shot / in-context learning
- Chain-of-Thought-style structured reasoning
- Tree-of-Thought-style multi-perspective reasoning

Using the same patient context across all methods will enable a controlled comparison of the resulting LLM outputs.